In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus()

import scanpy as sc
import plotnine as gg
import pandas as pd
import scipy.stats as st
import numpy as np
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]

adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")
adata_

In [ ]:
class MMDTest:
    def __init__(self, kernel_type="rbf", sigma=None):
        self.kernel_type = kernel_type
        self.sigma = sigma

    def k(self, X, Y):
        X_ = X[:, None]
        Y_ = Y[None, :]
        if self.kernel_type == "rbf":
            dists = ((X_ - Y_) ** 2).sum(-1)
            return np.exp(-dists / (2 * self.sigma**2))
        elif self.kernel_type == "linear":
            return (X_ * Y_).sum(-1)
        else:
            raise ValueError(f"Invalid kernel type: {self.kernel_type}")

    def _estimate_mmd(self, X, Y):
        K_XX = self.k(X, X)
        K_XX[np.diag_indices_from(K_XX)] = 0.0
        K_YY = self.k(Y, Y)
        K_YY[np.diag_indices_from(K_YY)] = 0.0
        K_XY = self.k(X, Y)

        contrib_xx = K_XX.sum() / (X.shape[0] * (X.shape[0] - 1))
        contrib_yy = K_YY.sum() / (Y.shape[0] * (Y.shape[0] - 1))
        contrib_xy = K_XY.sum() / (X.shape[0] * Y.shape[0])

        return contrib_xx + contrib_yy - 2 * contrib_xy

    def _preprocess(self, X, Y):
        if (self.kernel_type == "rbf") and (self.sigma is None):
            Z = np.concatenate([X, Y])
            dists = ((Z[:, None] - Z[None, :]) ** 2).sum(-1)
            upper_tri = dists[np.triu_indices_from(dists, k=1)]
            median_sq_dist = np.median(upper_tri)
            self.sigma = np.sqrt(median_sq_dist) if median_sq_dist > 0 else 1.0
        else:
            pass

    def test(self, X, Y, n_iter=10000):
        self._preprocess(X, Y)
        null_dist = []
        Z = np.concatenate([X, Y])
        m = X.shape[0]
        for _ in range(n_iter):
            indices = np.random.choice(np.arange(Z.shape[0]), size=Z.shape[0], replace=True)
            Z_boot = Z[indices]
            X_boot = Z_boot[:m]
            Y_boot = Z_boot[m:]
            null_dist.append(self._estimate_mmd(X_boot, Y_boot))

        null_dist = np.array(null_dist)
        p_value = (null_dist >= self._estimate_mmd(X, Y)).mean()
        return p_value

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from functools import partial


class MMDTestJax:
    def __init__(self, kernel_type="rbf", sigma=None, max_n=50):
        self.kernel_type = kernel_type
        self.sigma = sigma
        self.max_n = max_n  # Maximum expected sample size per group

    def k_matrix(self, X, Y):
        # Computes pairwise kernel matrix
        # Uses standard implementation that works with padded inputs
        # (padding usually results in dummy distances, masked out later)
        X_sq = jnp.sum(X**2, axis=1, keepdims=True)
        Y_sq = jnp.sum(Y**2, axis=1, keepdims=True)
        dists = X_sq + Y_sq.T - 2 * jnp.dot(X, Y.T)

        if self.kernel_type == "rbf":
            return jnp.exp(-dists / (2 * self.sigma**2))
        elif self.kernel_type == "linear":
            return jnp.dot(X, Y.T)
        else:
            raise ValueError(f"Invalid kernel type")

    def _estimate_mmd_masked(self, K_XY, m, n):
        # Create masks for valid data
        # K_XY is (N_total, N_total) where N_total = m + n (padded conceptually)

        # In the bootstrap, we treat the first m indices as X, next n as Y
        mask_x = jnp.arange(K_XY.shape[0]) < m
        mask_y = (jnp.arange(K_XY.shape[0]) >= m) & (jnp.arange(K_XY.shape[0]) < (m + n))

        # Expand masks for matrix operations
        MxMx = mask_x[:, None] * mask_x[None, :]
        MyMy = mask_y[:, None] * mask_y[None, :]
        MxMy = mask_x[:, None] * mask_y[None, :]

        # Zero out diagonals for unbiased estimator
        diag_mask = jnp.eye(K_XY.shape[0], dtype=bool)

        # Sums with masks
        sum_xx = jnp.sum(K_XY * MxMx * (~diag_mask))
        sum_yy = jnp.sum(K_XY * MyMy * (~diag_mask))
        sum_xy = jnp.sum(K_XY * MxMy)  # XY block doesn't involve self-pairs usually

        # Normalization
        # Use jnp.maximum to avoid division by zero if m=1 (though MMD requires m>1)
        c_xx = sum_xx / jnp.maximum(1.0, m * (m - 1))
        c_yy = sum_yy / jnp.maximum(1.0, n * (n - 1))
        c_xy = sum_xy / jnp.maximum(1.0, m * n)

        return c_xx + c_yy - 2 * c_xy

    @partial(jax.jit, static_argnames=["self"])
    def _compute_p_value_padded(self, keys, Z_padded, m, n):
        # 1. Compute Kernel on ALL data (padded)
        # Note: We compute K on 2*max_n size, but we only care about the top (m+n)x(m+n)
        # Optimization: We can just compute K on Z_padded once.
        K_Z = self.k_matrix(Z_padded, Z_padded)

        # Total valid samples
        N = m + n

        # Function to run one bootstrap iteration
        def body(key):
            # Sample N indices from range [0, N)
            # We use random.uniform and floor to avoid dynamic shape in random.choice
            rand_float = jax.random.uniform(key, shape=(self.max_n * 2,))
            idx = jnp.floor(rand_float * N).astype(jnp.int32)

            # Since we only need 'N' samples, but arrays must be static size 2*max_n:
            # We just use the sampled indices to gather from K_Z.
            # The mask in _estimate_mmd_masked will ignore the garbage at the end.

            # Resample Kernel
            K_boot = K_Z[idx][:, idx]

            # Calculate MMD on the first m (as X) and next n (as Y)
            return self._estimate_mmd_masked(K_boot, m, n)

        # Vectorize bootstrap
        null_dist = jax.vmap(body)(keys)

        # Compute observed statistic (no resampling)
        # The observed data corresponds to indices 0..m-1 (X) and m..m+n-1 (Y) of Z
        obs_stat = self._estimate_mmd_masked(K_Z, m, n)

        return (null_dist >= obs_stat).mean()

    def _preprocess(self, X, Y):
        # Heuristic for sigma (median heuristic) using numpy for safety/simplicity
        if (self.kernel_type == "rbf") and (self.sigma is None):
            # Downsample for sigma estimation if too large to avoid O(N^2)
            # (Optional optimization, here doing exact)
            Z = np.concatenate([X, Y])
            if Z.shape[0] > 2000:
                idx = np.random.choice(Z.shape[0], 2000, replace=False)
                Z = Z[idx]
            dists = ((Z[:, None] - Z[None, :]) ** 2).sum(-1)
            upper = dists[np.triu_indices_from(dists, k=1)]
            median = np.median(upper)
            self.sigma = np.sqrt(median) if median > 0 else 1.0

    def test(self, X, Y, n_iter=10000):
        # 1. Preprocess (Sigma) - run on CPU/Numpy to avoid JIT overhead
        self._preprocess(X, Y)

        m = X.shape[0]
        n = Y.shape[0]

        # 2. Pad Inputs to static size
        # Total buffer size needed is 2 * max_n (fit both X and Y max)
        target_size = 2 * self.max_n

        if m + n > target_size:
            raise ValueError(f"Input size {m+n} exceeds max_n capacity {target_size}")

        # Concatenate and Pad
        Z = np.concatenate([X, Y])
        pad_width = ((0, target_size - Z.shape[0]), (0, 0))
        Z_padded = np.pad(Z, pad_width, mode="constant")

        # 3. Jitted Execution
        keys = jax.random.split(jax.random.PRNGKey(42), n_iter)

        # Note: m and n are passed as dynamic args (tracers), not static.
        # Shapes of Z_padded are static.
        return float(self._compute_p_value_padded(keys, jnp.array(Z_padded), m, n))

In [ ]:
pert_name_x = "lpxB"
pert_name_y = "lpxD"

adata_x = adata[adata.obs["gene"] == pert_name_x]
adata_y = adata[adata.obs["gene"] == pert_name_y]
print(adata_x.X.shape, adata_y.X.shape)

X = adata_x.obsm["X_scVI"]
Y = adata_y.obsm["X_scVI"]

mmd_test = MMDTest(kernel_type="rbf", sigma=5)
start = time.time()
for _ in range(10):
    p_value = mmd_test.test(X, Y)
print("sigma: ", mmd_test.sigma)
print(p_value)
end = time.time()
print(f"Time taken: {end - start} seconds")

mmd_test_jax = MMDTestJax(kernel_type="rbf", sigma=5)
start = time.time()
for _ in range(10):
    p_value = mmd_test_jax.test(X, Y)
print("sigma: ", mmd_test_jax.sigma)
print(p_value)
end = time.time()
print(f"Time taken: {end - start} seconds")

In [ ]:
pert_name_x = "fur"
pert_name_y = "fldA"
# gene_1 = "fur"
# gene_2 = "fldA"


adata_x = adata[adata.obs["gene"] == pert_name_x]
adata_y = adata[adata.obs["gene"] == pert_name_y]
print(adata_x.X.shape, adata_y.X.shape)

X = adata_x.obsm["X_scVI"]
Y = adata_y.obsm["X_scVI"]

mmd_test = MMDTest(kernel_type="rbf", sigma=5)
start = time.time()
for _ in range(10):
    p_value = mmd_test.test(X, Y)
print("sigma: ", mmd_test.sigma)
print(p_value)
end = time.time()
print(f"Time taken: {end - start} seconds")

mmd_test_jax = MMDTestJax(kernel_type="rbf", sigma=5)
start = time.time()
for _ in range(10):
    p_value = mmd_test_jax.test(X, Y)
print("sigma: ", mmd_test_jax.sigma)
print(p_value)
end = time.time()
print(f"Time taken: {end - start} seconds")

In [ ]:
operon_df = pd.read_csv("/workspace/data/RegulonDB/TUSet.tsv", sep="\t", comment="#")
operon_df.columns = operon_df.columns.str.replace(r"^\d+\)", "", regex=True)
operon_df_filtered = operon_df.query("confidenceLevel == 'S'")

In [ ]:
operon_pairs = []
for i, row in tqdm(operon_df_filtered.iterrows()):
    genes_in_operon = row["tuGenes"][:-1].split(";")
    for pair_id in range(len(genes_in_operon) - 1):
        gene_1 = genes_in_operon[pair_id]
        gene_2 = genes_in_operon[pair_id + 1]
        operon_pairs.append((gene_1, gene_2))
operon_pairs = (
    pd.DataFrame(operon_pairs, columns=["gene_1", "gene_2"])
    .assign(gene_key=lambda x: x.gene_1 + "_" + x.gene_2)
    .drop_duplicates("gene_key", keep="first")
)
operon_pairs

In [ ]:
results_df = []
mmd_test = MMDTestJax(kernel_type="rbf", sigma=5)
for i, pair in tqdm(operon_pairs.iterrows()):
    pert_name_x = pair["gene_1"]
    pert_name_y = pair["gene_2"]
    adata_x = adata[adata.obs["gene"] == pert_name_x]
    n_1 = adata_x.shape[0]
    adata_y = adata[adata.obs["gene"] == pert_name_y]
    n_2 = adata_y.shape[0]
    both_present = n_1 >= 1 and n_2 >= 1

    if both_present:
        X = adata_x.obsm["X_scVI"][:50]
        Y = adata_y.obsm["X_scVI"][:50]
        p_value = mmd_test.test(X, Y)
    else:
        p_value = None
    results_df.append(
        {
            "gene_1": pert_name_x,
            "gene_2": pert_name_y,
            "n_1": n_1,
            "n_2": n_2,
            "both_present": both_present,
            "p_value": p_value,
        }
    )
results_df = pd.DataFrame(results_df)

In [ ]:
sorted_res = results_df.sort_values(by="p_value", ascending=True).head(50)
sorted_res["p_value"].hist()
plt.show()

In [ ]:
sorted_res

In [ ]:
# gene_1 = "dnaK"
# gene_2 = "dnaJ"

# gene_1 = "ychA"
# gene_2 = "ychQ"

gene_1 = "ribE"
gene_2 = "ribD"

# idx = 0
# gene_1 = sorted_res.iloc[idx]["gene_1"]
# gene_2 = sorted_res.iloc[idx]["gene_2"]

plot_df = adata.obs
plot_df["UMAP1"] = adata.obsm["X_umap"][:, 0]
plot_df["UMAP2"] = adata.obsm["X_umap"][:, 1]

plot_df_subset = plot_df[plot_df["gene"].isin([gene_1, gene_2])].copy()
plot_df_subset["gene"] = plot_df_subset["gene"].astype(str)

fig = (
    gg.ggplot(plot_df, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point(alpha=0.5)
    + gg.geom_point(plot_df_subset, gg.aes(x="UMAP1", y="UMAP2", color="gene"), alpha=1)
)
display(fig)

plot_df_case = adata_case.obs.copy()
plot_df_case_subset = plot_df_case[
    plot_df_case["gene"].isin(["ribA", gene_1, gene_2, "ribF", "ribH"])
].copy()
plot_df_case_subset["gene"] = plot_df_case_subset["gene"].astype(str)

fig = (
    gg.ggplot(plot_df_case, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"))
    + gg.geom_point(alpha=0.5)
    + gg.geom_point(
        plot_df_case_subset,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
        alpha=1,
    )
)
display(fig)